# 🚀 AI YouTube Studio — Colab GPU Worker [NODE 3 — CHẠY SONG SONG]
### Trạm Render GPU NVIDIA T4/A100 Siêu Tốc (20-30s/Video) — Tự Động Quét & Hoàn Thiện Video Từ Telegram & Google Drive!

---
### 📌 HƯỚNG DÙNG 1-CHẠM TRÊN ĐIỆN THOẠI HOẶC MÁY TÍNH:
1. **Bấm Chạy Tất Cả:** Vào menu **Runtime** (Thời gian chạy) ➡️ Bấm **Run all** (Chạy tất cả).
2. **Cấp quyền Drive:** Bấm **Connect to Google Drive** (khi hiện cửa sổ hỏi).
3. **TẮT MÀN HÌNH HOẶC THOÁT LINK ĐƯỢC NGAY:**
   - Colab sẽ tự động quét toàn bộ kịch bản do Bot Telegram gửi lên Drive (`/AI_YouTube_RenderFarm/pending/`).
   - Tự động hỏi và xử lý nốt các video dở dang (nếu có từ phiên trước).
   - Sinh giọng đọc Edge-TTS + khớp phụ đề WhisperX GPU + tải stock B-Roll 9:16 + render NVENC chỉ trong 20-30 giây.
   - Video hoàn thành tự động lưu vào `/AI_YouTube_RenderFarm/completed/` để Bot lấy về gửi cho bạn!


In [ ]:
# [BƯỚC 1] Kiểm tra sức mạnh cạc đồ họa GPU & Hạn mức phiên làm việc
import subprocess

print('=' * 65)
print('🚀 AI YOUTUBE STUDIO — KIỂM TRA HẠN MỨC GPU WORKER')
print('=' * 65)

try:
    gpu_info = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name,memory.total,utilization.gpu', '--format=csv,noheader']
    ).decode('utf-8').strip()
    print(f'✅ GPU ĐANG KẾT NỐI: {gpu_info}')
    print('✨ Tình trạng: Sẵn sàng render video siêu tốc 20-30s/video!')
    print('💡 Lưu ý: Colab chỉ dùng GPU máy ảo dựng video, KHÔNG tốn hạn mức Gemini Ultra.')
except Exception:
    print('⚠️ CẢNH BÁO: Phiên Colab này chưa có cạc GPU (đang chạy bằng CPU)!')
    print('   👉 1. Vào menu: Runtime (Thời gian chạy) ➡️ Change runtime type ➡️ Chọn T4 GPU.')
    print('   👉 2. Nếu Google báo hết hạn mức GPU hôm nay: Hãy bấm sang Link Tài Khoản 2 hoặc 3!')

print('=' * 65)
!nvidia-smi


In [ ]:
# [BƯỚC 2] Cài đặt các thư viện siêu tốc cho GPU (WhisperX, Edge-TTS, FFmpeg NVENC)
!pip install -q edge-tts faster-whisper pillow requests pydantic python-dotenv
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1
print("✅ Đã cài đặt xong môi trường GPU Worker Siêu Tốc!")

In [ ]:
# [BƯỚC 3] Kết nối Google Drive & Thiết lập Không Gian Render Farm (Hỗ trợ xoay 3 Tài Khoản)
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print(f"ℹ️ Lỗi kết nối Google Drive: {e}")

# Tự động phát hiện thư mục Render Farm trên MyDrive hoặc Shared with me
base_drive = Path('/content/drive')
candidate = base_drive / 'MyDrive' / 'AI_YouTube_RenderFarm'

if not candidate.exists():
    # Quét tìm nếu được tạo dưới dạng lối tắt hoặc thư mục chia sẻ
    found = list(base_drive.glob('**/AI_YouTube_RenderFarm'))
    if found:
        candidate = found[0]

DRIVE_FARM_DIR = candidate
PENDING_DIR = DRIVE_FARM_DIR / 'pending'
IN_PROGRESS_DIR = DRIVE_FARM_DIR / 'in_progress'
COMPLETED_DIR = DRIVE_FARM_DIR / 'completed'
ARCHIVE_DIR = DRIVE_FARM_DIR / 'archive'

if DRIVE_FARM_DIR.exists() and (candidate / 'pending').exists():
    for d in [PENDING_DIR, IN_PROGRESS_DIR, COMPLETED_DIR, ARCHIVE_DIR]:
        d.mkdir(parents=True, exist_ok=True)
    print('✅ ĐÃ KẾT NỐI VÀO KHO DỮ LIỆU RENDER FARM DÙNG CHUNG!')
    print(f'   📁 Thư mục làm việc:                 {DRIVE_FARM_DIR}')
    print(f'   📁 Hàng đợi chờ render (pending):     {PENDING_DIR}')
    print(f'   📁 Video đang dựng (in_progress):   {IN_PROGRESS_DIR}')
    print(f'   📁 Video hoàn thành (completed):     {COMPLETED_DIR}')
else:
    print('=' * 70)
    print('🚨 CẢNH BÁO: CHƯA KẾT NỐI VÀO THƯ MỤC AI_YouTube_RenderFarm CHUNG!')
    print('=' * 70)
    print('👉 NẾU ĐÂY LÀ TÀI KHOẢN PHỤ (TK 2 HOẶC TK 3):')
    print('   1. Mở https://drive.google.com bằng tài khoản Google này.')
    print('   2. Vào mục Được chia sẻ với tôi (Shared with me) ở cột bên trái.')
    print('   3. Chuột phải vào thư mục AI_YouTube_RenderFarm ➡️ Chọn Thêm lối tắt vào Drive (Add shortcut to Drive) ➡️ Chọn Drive của tôi (My Drive).')
    print('   4. Bấm chạy lại Bước 3 này là xong!')
    print('👉 NẾU ĐÂY LÀ TÀI KHOẢN CÁ NHÂN KHÁC (NGOÀI CỤM 3 TÀI KHOẢN):')
    print('   Vui lòng đổi sang 1 trong 3 tài khoản hệ thống:')
    print('   • TK 1: pool1.studio@example.com')
    print('   • TK 2: pool2.studio@example.com')
    print('   • TK 3: pool3.studio@example.com')
    print('=' * 70)


In [ ]:
# [BƯỚC 4] Bộ Máy Dựng Video GPU Toàn Năng (Full Render Engine)
import os
import re
import json
import time
import shutil
import asyncio
import urllib.parse
import requests
import subprocess
from pathlib import Path
from PIL import Image

def get_gpu_encoder():
    """Kiểm tra xem hệ thống có hỗ trợ bộ mã hóa phần cứng h264_nvenc không"""
    try:
        res = subprocess.run(["ffmpeg", "-encoders"], capture_output=True, text=True)
        if "h264_nvenc" in res.stdout:
            return "h264_nvenc"
    except Exception:
        pass
    return "libx264"

ENCODER = get_gpu_encoder()
print(f"🎮 Bộ mã hóa Video đang sử dụng: {ENCODER} ({'GPU Tăng Tốc 20s' if ENCODER == 'h264_nvenc' else 'CPU'})")

async def generate_voice_edge_tts(text: str, voice: str, output_path: str) -> bool:
    """Sinh giọng đọc tự nhiên bằng Edge-TTS trong 1-2 giây"""
    import edge_tts
    clean_text = re.sub(r'[*_#`~]', '', text).strip()
    if not clean_text:
        return False
    communicate = edge_tts.Communicate(clean_text, voice)
    await communicate.save(output_path)
    return Path(output_path).exists() and Path(output_path).stat().st_size > 1000

def transcribe_with_whisper(audio_path: str, language: str = None):
    """Dùng Faster-Whisper trên CUDA GPU để lấy timestamp chuẩn từng từ trong 2s, khóa ngôn ngữ chuẩn xác"""
    try:
        from faster_whisper import WhisperModel
        model = WhisperModel("small", device="cuda", compute_type="float16")
        transcribe_kwargs = {"word_timestamps": True}
        if language:
            transcribe_kwargs["language"] = language
        segments, info = model.transcribe(audio_path, **transcribe_kwargs)
        words = []
        for seg in segments:
            for w in seg.words:
                words.append({"word": w.word.strip(), "start": w.start, "end": w.end})
        return words
    except Exception as e:
        print(f"   ⚠️ WhisperX GPU fallback: {e}")
        return []

def create_dynamic_ass_subtitles(words: list, output_path: str, width: int = 1080, height: int = 1920, total_duration: float = 30.0, is_vietnamese: bool = False, custom_ass_content: str = None):
    """Tạo file phụ đề ASS kinetic karaoke {\kf} chuẩn P2.2 kèm CTA Banner"""
    if custom_ass_content and len(custom_ass_content.strip()) > 50:
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(custom_ass_content)
        return output_path

    header = f"""[Script Info]
ScriptType: v4.00+
PlayResX: {width}
PlayResY: {height}
ScaledBorderAndShadow: yes

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Default,Arial,52,&H0000FFFF,&H00FFFFFF,&H00000000,&H80000000,-1,0,0,0,100,100,0,0,1,5,2,2,40,40,240,1
Style: CTABadge,Arial,44,&H0000FFFF,&H000000FF,&H00000000,&H80000000,-1,0,0,0,100,100,0,0,1,5,3,2,40,40,750,1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
"""
    def fmt_time(sec):
        h = int(sec // 3600)
        m = int((sec % 3600) // 60)
        s = int(sec % 60)
        cs = int(round((sec - int(sec)) * 100))
        return f"{h:01d}:{m:02d}:{s:02d}.{cs:02d}"

    events = []
    chunk_size = 3
    for i in range(0, len(words), chunk_size):
        chunk = words[i:i+chunk_size]
        if not chunk:
            continue
        start_t = fmt_time(chunk[0]["start"])
        end_t = fmt_time(chunk[-1]["end"] + 0.15)
        # Tạo chuỗi kinetic karaoke {\kf} mượt mà theo từng âm tiết
        k_parts = []
        for w in chunk:
            dur_cs = max(1, int(round((w["end"] - w["start"]) * 100)))
            clean_word = w["word"].strip()
            k_parts.append(f"{{\kf{dur_cs}}}{clean_word}")
        line_text = " ".join(k_parts)
        events.append(f"Dialogue: 0,{start_t},{end_t},Default,,0,0,0,,{line_text}")

    # Hiển thị Animated CTA Banner ở 4.5 giây cuối video
    if total_duration > 4.0:
        cta_start = fmt_time(max(0.0, total_duration - 4.5))
        cta_end = fmt_time(total_duration)
        cta_text = "🔔 ĐĂNG KÝ KÊNH ĐÓN XEM PHẦN 2 🔔" if is_vietnamese else "🔔 SUBSCRIBE FOR PART 2 🔔"
        events.append(f"Dialogue: 1,{cta_start},{cta_end},CTABadge,,0,0,0,,{{\fade(250,150)\t(0,250,\fscx110\fscy110)\t(250,500,\fscx100\fscy100)}}{cta_text}")

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(header + "
".join(events) + "
")
    return output_path

def apply_ken_burns_to_image(img_path: str, out_path: str, duration: float, scene_id: int = 1):
    """Áp dụng chuyển động điện ảnh Ken Burns P2.1 Cubic Ease-Out 1080x1920 30fps bằng GPU NVENC"""
    total_frames = max(30, int(duration * 30))
    d = total_frames
    if scene_id == 1:
        # Cú Punch-in Zoom kịch tính cho Scene 1 (0-3s) khóa chặt sự chú ý
        vf_zoom = f"scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920,zoompan=z='1.05+(1.38-1.05)*(1-pow(1-min(on/{d}\,1)\,3))':x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':d={d}:s=1080x1920:fps=30,setsar=1"
    elif scene_id % 2 == 1:
        # Cubic ease-out Zoom In (1.0 -> 1.25)
        vf_zoom = f"scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920,zoompan=z='1.0+(1.25-1.0)*(1-pow(1-min(on/{d}\,1)\,2.8))':x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':d={d}:s=1080x1920:fps=30,setsar=1"
    else:
        # Cubic ease-out Zoom Out (1.25 -> 1.0)
        vf_zoom = f"scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920,zoompan=z='1.25-(1.25-1.0)*(1-pow(1-min(on/{d}\,1)\,2.8))':x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':d={d}:s=1080x1920:fps=30,setsar=1"
    cmd = [
        "ffmpeg", "-y",
        "-loop", "1", "-i", img_path,
        "-vf", vf_zoom,
        "-t", str(duration),
        "-c:v", ENCODER,
        "-preset", "fast",
        "-pix_fmt", "yuv420p",
        "-an",
        out_path
    ]
    subprocess.run(cmd, check=True)
    return True

def search_and_download_scene_media(query: str, visual_prompt: str, out_path: str, duration: float, pexels_key: str = None, pixabay_key: str = None, scene_id: int = 1):
    """
    Quy trình phân giải tài nguyên thị giác đa tầng (Multi-Tier Media Pipeline):
    Tier 1: Video dọc Pexels (FHD 1080x1920).
    Tier 2: Ảnh chụp chất lượng cao Pexels (Portrait Photo) -> Ken Burns.
    Tier 3: Pixabay Video & Photo (Động vật, thiên nhiên, vũ trụ) -> Ken Burns.
    Tier 4: Sinh ảnh siêu thực FLUX.1 (Pollinations AI) theo visual_prompt -> Zoompan Ken Burns.
    Tuyệt đối KHÔNG BAO GIỜ sinh ra khung hình màu đen/trơn đơn điệu!
    """
    clean_q = " ".join(re.sub(r'[^a-zA-Z0-9 ]', '', query or "").split()[:3])
    if not clean_q:
        clean_q = "wildlife nature"

    # Tier 1: Pexels Video
    if pexels_key:
        try:
            headers = {"Authorization": pexels_key}
            url = f"https://api.pexels.com/videos/search?query={clean_q}&orientation=portrait&per_page=3"
            res = requests.get(url, headers=headers, timeout=8)
            if res.status_code == 200:
                vids = res.json().get("videos", [])
                for v in vids:
                    files = v.get("video_files", [])
                    portrait = [f for f in files if f.get("width", 0) <= f.get("height", 0) and "mp4" in f.get("file_type", "mp4")]
                    if portrait:
                        ideal = [f for f in portrait if 720 <= f.get("height", 0) <= 1920]
                        chosen = ideal[0] if ideal else portrait[0]
                        link = chosen.get("link")
                        if link:
                            temp_raw = f"/tmp/raw_px_{time.time()}.mp4"
                            r_down = requests.get(link, timeout=25)
                            if r_down.status_code == 200 and len(r_down.content) > 5000:
                                with open(temp_raw, "wb") as f_raw:
                                    f_raw.write(r_down.content)
                                cmd = [
                                    "ffmpeg", "-y",
                                    "-stream_loop", "-1",
                                    "-i", temp_raw,
                                    "-t", str(duration),
                                    "-vf", "scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920,setsar=1,fps=30",
                                    "-c:v", ENCODER,
                                    "-preset", "fast",
                                    "-pix_fmt", "yuv420p",
                                    "-an",
                                    out_path
                                ]
                                subprocess.run(cmd, check=True)
                                Path(temp_raw).unlink(missing_ok=True)
                                print(f"   🎬 [Pexels Video] Đã tải clip cho '{clean_q}'")
                                return True
        except Exception:
            pass

    # Tier 2: Pexels Photos (Ảnh chất lượng cực cao -> Ken Burns)
    if pexels_key:
        try:
            headers = {"Authorization": pexels_key}
            url = f"https://api.pexels.com/v1/search?query={clean_q}&orientation=portrait&per_page=3"
            res = requests.get(url, headers=headers, timeout=8)
            if res.status_code == 200:
                photos = res.json().get("photos", [])
                if photos:
                    img_url = photos[0].get("src", {}).get("large2x") or photos[0].get("src", {}).get("large")
                    if img_url:
                        temp_img = f"/tmp/px_img_{time.time()}.jpg"
                        r_img = requests.get(img_url, timeout=20)
                        if r_img.status_code == 200 and len(r_img.content) > 5000:
                            with open(temp_img, "wb") as fi:
                                fi.write(r_img.content)
                            apply_ken_burns_to_image(temp_img, out_path, duration, scene_id)
                            Path(temp_img).unlink(missing_ok=True)
                            print(f"   📸 [Pexels Photo] Đã áp dụng Ken Burns cho '{clean_q}'")
                            return True
        except Exception:
            pass

    # Tier 3: Pixabay API
    if pixabay_key:
        try:
            url_pix = f"https://pixabay.com/api/?key={pixabay_key}&q={urllib.parse.quote(clean_q)}&image_type=photo&orientation=vertical&per_page=3"
            res_pix = requests.get(url_pix, timeout=8)
            if res_pix.status_code == 200:
                hits = res_pix.json().get("hits", [])
                if hits:
                    img_url = hits[0].get("largeImageURL") or hits[0].get("webformatURL")
                    if img_url:
                        temp_img = f"/tmp/pix_img_{time.time()}.jpg"
                        r_img = requests.get(img_url, timeout=20)
                        if r_img.status_code == 200 and len(r_img.content) > 5000:
                            with open(temp_img, "wb") as fi:
                                fi.write(r_img.content)
                            apply_ken_burns_to_image(temp_img, out_path, duration, scene_id)
                            Path(temp_img).unlink(missing_ok=True)
                            print(f"   📸 [Pixabay] Đã áp dụng Ken Burns cho '{clean_q}'")
                            return True
        except Exception:
            pass

    # Tier 4: AI Image Generation via Pollinations FLUX (Không giới hạn, 9:16 siêu thực 768x1344)
    try:
        prompt_use = visual_prompt if (visual_prompt and len(visual_prompt) > 10) else f"cinematic 9:16 vertical shot of {clean_q}, hyper-detailed, photorealistic, 8k, dramatic lighting"
        clean_prompt = " ".join(prompt_use.split()[:50])
        encoded = urllib.parse.quote(clean_prompt)
        seed = int(time.time() * 1000) % 999999 + scene_id
        flux_url = f"https://image.pollinations.ai/prompt/{encoded}?width=768&height=1344&model=flux&nologo=true&seed={seed}"
        temp_img = f"/tmp/flux_{time.time()}_{scene_id}.jpg"
        r_flux = requests.get(flux_url, timeout=18)
        if r_flux.status_code == 200 and len(r_flux.content) > 5000:
            with open(temp_img, "wb") as ff:
                ff.write(r_flux.content)
            apply_ken_burns_to_image(temp_img, out_path, duration, scene_id)
            Path(temp_img).unlink(missing_ok=True)
            print(f"   🎨 [FLUX AI Image] Đã sinh ảnh siêu thực và tạo motion clip cho Cảnh {scene_id}!")
            return True
    except Exception as e_flux:
        print(f"   ⚠️ Lỗi FLUX fallback: {e_flux}")

    # Fallback cuối cùng nếu mất mạng: tạo hình nền nghệ thuật có chuyển động
    temp_img = f"/tmp/bg_{scene_id}.jpg"
    colors = ["#0d1117", "#161b22", "#1f242c", "#0f172a", "#1e1b4b", "#2e1065", "#1c1917"]
    c_pick = colors[scene_id % len(colors)]
    cmd_img = ["ffmpeg", "-y", "-f", "lavfi", "-i", f"color=c={c_pick}:s=1080x1920:d=1", "-frames:v", "1", temp_img]
    subprocess.run(cmd_img, check=True)
    apply_ken_burns_to_image(temp_img, out_path, duration, scene_id)
    Path(temp_img).unlink(missing_ok=True)
    return True


In [ ]:
# [BƯỚC 5] QUÉT & HỎI Ý KIẾN NGƯỜI DÙNG VỀ CÁC VIDEO DỞ DANG / HÀNG ĐỢI
import time
import sys
import select

print("=" * 75)
print("🔍 ĐANG QUÉT CÁC TÁC VỤ KỊCH BẢN TỪ BOT TELEGRAM TRÊN GOOGLE DRIVE...")
print("=" * 75)

pending_tasks = sorted(list(PENDING_DIR.glob("task_*.json")))
in_progress_tasks = sorted(list(IN_PROGRESS_DIR.glob("task_*.json")))

print(f"📊 KẾT QUẢ QUÉT HÀNG ĐỢI:")
print(f"   • 📥 Video MỚI đang chờ:       {len(pending_tasks)} video")
for t in pending_tasks:
    try:
        d = json.load(open(t, encoding='utf-8'))
        print(f"     └─ [Job #{d.get('job_id')}] {d.get('title', 'Không có tiêu đề')} ({len(d.get('scenes', []))} cảnh)")
    except Exception:
        print(f"     └─ {t.name}")

print(f"   • ⚠️ Video DỞ DANG từ phiên trước: {len(in_progress_tasks)} video")
for t in in_progress_tasks:
    try:
        d = json.load(open(t, encoding='utf-8'))
        print(f"     └─ [Job #{d.get('job_id')}] {d.get('title', 'Không có tiêu đề')} (Bị ngắt giữa chừng)")
    except Exception:
        print(f"     └─ {t.name}")

tasks_to_run = []

if not pending_tasks and not in_progress_tasks:
    print("\n☕ Hiện tại không có video nào cần dựng. Colab sẽ chuyển sang chế độ Túc Trực (Auto-Listen)!")
else:
    print("-" * 75)
    print("❓ BẠN MUỐN CHẠY NHỮNG VIDEO NÀO?")
    print("   [1] 👉 Chạy TẤT CẢ (Gồm cả video mới và video dở dang) [KHUYÊN DÙNG]")
    print("   [2] 👉 Chỉ chạy các video MỚI đang chờ")
    print("   [3] 👉 Chỉ chạy lại các video DỞ DANG từ phiên trước")
    print("⏳ Tự động chọn [1] sau 10 giây nếu bạn đã tắt màn hình điện thoại / không thao tác...")
    
    # Bộ đếm ngược 10 giây tự động
    choice = "1"
    try:
        import threading
        user_input = []
        def get_inp():
            try:
                val = input("👉 Nhập lựa chọn [1, 2, 3] (mặc định 1): ").strip()
                if val:
                    user_input.append(val)
            except Exception:
                pass
        th = threading.Thread(target=get_inp)
        th.daemon = True
        th.start()
        th.join(timeout=10.0)
        if user_input:
            choice = user_input[0]
            print(f"\n👉 Bạn đã chọn: [{choice}]")
        else:
            print("\n⏰ Hết 10s: Tự động chọn [1] Chạy TẤT CẢ để bạn không phải chờ!")
    except Exception:
        choice = "1"

    if choice == "2":
        tasks_to_run = pending_tasks
    elif choice == "3":
        tasks_to_run = in_progress_tasks
    else:
        tasks_to_run = in_progress_tasks + pending_tasks

    print(f"🚀 Danh sách chuẩn bị sản xuất bằng GPU: {len(tasks_to_run)} video.")


In [ ]:
# [BƯỚC 6] THỰC THI SẢN XUẤT VIDEO SIÊU TỐC BẰNG GPU & ĐẨY VỀ DRIVE (HỖ TRỢ ĐA WORKER SONG SONG)
import time
import os
import shutil
import subprocess
from pathlib import Path

async def process_single_task(task_path: Path):
    t_start = time.time()
    active_task_path = IN_PROGRESS_DIR / task_path.name

    # KHÓA TÁC VỤ NGUYÊN TỬ (ATOMIC CLAIM): Tránh xung đột khi nhiều Worker Colab Pro+ cùng chạy
    try:
        if not task_path.exists():
            return False
        shutil.move(str(task_path), str(active_task_path))
    except Exception:
        print(f"ℹ️ Tác vụ {task_path.name} đã được một Worker GPU khác nhận trước. Bỏ qua.")
        return False

    try:
        with open(active_task_path, "r", encoding="utf-8") as f:
            task = json.load(f)
    except Exception as e:
        print(f"❌ Lỗi đọc task {active_task_path.name}: {e}")
        return False

    job_id = task.get("job_id", int(time.time()))
    title = task.get("title", f"Video_{job_id}")
    voice = task.get("voice", "en-US-AndrewNeural")
    narration = task.get("full_narration", "")
    scenes = task.get("scenes", [])
    pexels_key = task.get("pexels_api_key")
    pixabay_key = task.get("pixabay_api_key", "57761912-c9510c981436c9c9ff731f4e5")

    full_text = narration or " ".join([s.get("narration", "") for s in scenes])

    # Khớp ngôn ngữ chuẩn xác giữa nội dung văn bản và giọng đọc AI
    is_vietnamese = bool(re.search(r'[àáảãạăắằẳẵặâấầẩẫậèéẻẽẹêếềểễệìíỉĩịòóỏõọôốồổỗộơớờởỡợùúủũụưứừửữựỳýỷỹỵđ]', full_text, re.IGNORECASE))
    if is_vietnamese and not voice.lower().startswith("vi-"):
        voice = "vi-VN-NamMinhNeural"
    elif not is_vietnamese and voice.lower().startswith("vi-"):
        voice = "en-US-ChristopherNeural"

    whisper_lang = "vi" if is_vietnamese else "en"

    print("=" * 75)
    print(f"🎬 [BẮT ĐẦU DỰNG] Video #{job_id}: '{title}'")
    print(f"   🎙️ Giọng đọc: {voice} (Ngôn ngữ: {whisper_lang.upper()}) | 📝 Số phân cảnh: {len(scenes)}")

    work_dir = Path(f"/tmp/job_{job_id}")
    work_dir.mkdir(parents=True, exist_ok=True)
    audio_path = str(work_dir / f"audio_{job_id}.mp3")
    sub_path = str(work_dir / f"sub_{job_id}.ass")
    local_render_mp4 = work_dir / f"rendered_{job_id}.mp4"
    out_mp4 = COMPLETED_DIR / f"video_{job_id}.mp4"

    # 1. Sinh Audio qua Edge-TTS
    print("   1️⃣ Đang tạo giọng đọc AI...")
    ok_voice = await generate_voice_edge_tts(full_text, voice, audio_path)
    if not ok_voice:
        print(f"❌ Lỗi sinh âm thanh cho #{job_id}")
        return False

    dur_res = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", audio_path], capture_output=True, text=True)
    total_duration = float(dur_res.stdout.strip() or 30.0)
    print(f"   ✅ Giọng đọc hoàn tất: {total_duration:.1f}s")

    # 2. Khớp phụ đề bằng WhisperX trên GPU với ngôn ngữ chỉ định
    print(f"   2️⃣ Đang khớp phụ đề động từng từ bằng GPU (Language: {whisper_lang})...")
    sub_content_task = task.get("subtitle_ass_content")
    if sub_content_task:
        create_dynamic_ass_subtitles([], sub_path, custom_ass_content=sub_content_task)
        print("   ✅ [P2.2 Sync] Sử dụng file phụ đề ASS Kinetic Karaoke chuẩn P2 từ Server!")
    else:
        words = transcribe_with_whisper(audio_path, language=whisper_lang)
        if words:
            create_dynamic_ass_subtitles(words, sub_path, total_duration=total_duration, is_vietnamese=is_vietnamese)
            print(f"   ✅ Phụ đề ASS kinetic karaoke (\\kf) & CTA Banner sẵn sàng ({len(words)} từ)!")
        else:
            sub_path = None

    # 3. Chuẩn bị Scene Clips (Multi-Tier Media Engine)
    print(f"   3️⃣ Đang xử lý {len(scenes)} phân cảnh video 9:16...")
    scene_clips = []
    time_per_scene = total_duration / max(len(scenes), 1)

    # Kiểm tra gói assets đã được Server duyệt qua AI Vision (Pre-baked Bundle)
    prebaked_dir = DRIVE_FARM_DIR / "pending" / f"assets_{job_id}"
    has_prebaked = prebaked_dir.exists() and any(prebaked_dir.glob("scene_*.*"))

    for idx, sc in enumerate(scenes):
        s_id = sc.get("scene_id", idx + 1)
        s_clip = str(work_dir / f"scene_{s_id:03d}.mp4")
        dur_s = float(sc.get("estimated_seconds") or time_per_scene)

        # Ưu tiên 1: Lấy asset đã qua thẩm định AI Vision của Server
        matched_prebaked_clip = list(prebaked_dir.glob(f"scene_{s_id:03d}.mp4")) if has_prebaked else []
        matched_prebaked_img = list(prebaked_dir.glob(f"scene_{s_id:03d}.*")) if has_prebaked else []

        if matched_prebaked_clip:
            shutil.copy2(str(matched_prebaked_clip[0]), s_clip)
            print(f"   🛡️ [AI Vision Verified] Cảnh {s_id}: Dùng video clip đã kiểm duyệt từ Server!")
            scene_clips.append(s_clip)
            continue
        elif matched_prebaked_img and not matched_prebaked_img[0].suffix.endswith(".mp4"):
            apply_ken_burns_to_image(str(matched_prebaked_img[0]), s_clip, dur_s, s_id)
            print(f"   🛡️ [AI Vision Verified] Cảnh {s_id}: Dùng ảnh đã kiểm duyệt từ Server + Ken Burns P2!")
            scene_clips.append(s_clip)
            continue

        # Ưu tiên 2: Tự động tìm kiếm Stock Media với Ken Burns P2.1
        q = sc.get("stock_query") or sc.get("visual_keyword") or title
        v_prompt = sc.get("visual_prompt") or q
        search_and_download_scene_media(
            query=q,
            visual_prompt=v_prompt,
            out_path=s_clip,
            duration=dur_s,
            pexels_key=pexels_key,
            pixabay_key=pixabay_key,
            scene_id=s_id
        )
        scene_clips.append(s_clip)

    # 4. Ghép Clips & Render hoàn chỉnh bằng GPU vào ổ đĩa nội bộ /tmp
    print(f"   4️⃣ Đang lắp ráp và kết xuất video bằng {ENCODER} GPU...")
    concat_txt = str(work_dir / "concat.txt")
    with open(concat_txt, "w", encoding="utf-8") as f_c:
        for sc_p in scene_clips:
            f_c.write(f"file '{sc_p}'\n")

    silent_mp4 = str(work_dir / "silent.mp4")
    subprocess.run([
        "ffmpeg", "-y",
        "-f", "concat", "-safe", "0", "-i", concat_txt,
        "-c:v", ENCODER,
        "-preset", "fast",
        "-pix_fmt", "yuv420p",
        "-an",
        silent_mp4
    ], check=True)

    mux_args = [
        "ffmpeg", "-y",
        "-i", silent_mp4,
        "-i", audio_path,
    ]
    if sub_path and Path(sub_path).exists():
        mux_args.extend(["-vf", f"ass={sub_path}"])
    mux_args.extend([
        "-af", "loudnorm=I=-14:TP=-1.5:LRA=11",
        "-c:v", ENCODER,
        "-preset", "fast",
        "-c:a", "aac",
        "-b:a", "192k",
        "-shortest",
        str(local_render_mp4)
    ])
    subprocess.run(mux_args, check=True)

    # 5. Lưu trữ video sang Google Drive an toàn
    if local_render_mp4.exists() and local_render_mp4.stat().st_size > 1000:
        print(f"   💾 Đang lưu video lên Google Drive ({local_render_mp4.stat().st_size / (1024*1024):.1f} MB)...\n")
        shutil.copy2(str(local_render_mp4), str(out_mp4))
        try:
            os.sync()
        except Exception:
            pass
    else:
        print(f"❌ File render bị lỗi hoặc rỗng cho #{job_id}")
        return False

    cost_time = time.time() - t_start
    meta = {
        "job_id": job_id,
        "title": title,
        "duration_seconds": total_duration,
        "render_time_seconds": round(cost_time, 1),
        "encoder": ENCODER,
        "completed_at": time.strftime('%Y-%m-%d %H:%M:%S'),
        "file_name": f"video_{job_id}.mp4"
    }
    with open(COMPLETED_DIR / f"meta_{job_id}.json", "w", encoding="utf-8") as f_m:
        json.dump(meta, f_m, ensure_ascii=False, indent=2)
    try:
        os.sync()
    except Exception:
        pass

    shutil.move(str(active_task_path), str(ARCHIVE_DIR / active_task_path.name))
    shutil.rmtree(str(work_dir), ignore_errors=True)

    # 6. Bắn thông báo tức thì về Telegram từ Colab
    bot_token = task.get("telegram_bot_token")
    chat_ids = task.get("telegram_chat_ids", [])
    if bot_token and chat_ids:
        for cid in chat_ids:
            try:
                requests.post(
                    f"https://api.telegram.org/bot{bot_token}/sendMessage",
                    json={
                        "chat_id": cid,
                        "text": f"🎉 *[COLAB GPU FARM] VIDEO #{job_id} ĐÃ RENDER HOÀN TẤT!*\n\n📌 *Tiêu đề:* {title}\n⚡ *Thời gian:* {cost_time:.1f}s (NVENC GPU)\n☁️ Đã lưu an toàn lên Google Drive. Bot Telegram đang tải về gửi cho nhóm!",
                        "parse_mode": "Markdown"
                    },
                    timeout=5
                )
            except Exception:
                pass

    print("=" * 75)
    print(f"🎉 THÀNH CÔNG! VIDEO #{job_id} ĐÃ HOÀN TẤT TRONG {cost_time:.1f} GIÂY!")
    print(f"📁 File đã lưu vào Google Drive: {out_mp4}")
    print(f"🤖 Bot Telegram sẽ tự động quét thấy và gửi video cho bạn!")
    print("=" * 75)
    return True

# ──────────────────────────────────────────────────────────────────────────────
# TÙY CHỌN DÀNH CHO COLAB PRO+ (CHẠY NGẦM 24H LIÊN TỤC TRÊN GOOGLE DRIVE)
# ──────────────────────────────────────────────────────────────────────────────
RUN_CONTINUOUS_DAEMON = True # True = Túc trực chạy ngầm 24h (Colab Pro+); False = Render xong tự ngắt

async def run_worker_daemon():
    print("=" * 75)
    print("🚀 BẮT ĐẦU VÒNG LẶP SẢN XUẤT GPU WORKER (PRO+ 24H BACKGROUND)")
    print(f"   ⚙️ Chế độ Túc Trực (Daemon Mode): {'BẬT (Colab Pro+ chạy ngầm 24h)' if RUN_CONTINUOUS_DAEMON else 'TẮT (Chạy 1 lượt rồi dừng)'}")
    print("=" * 75)

    # 1. Chạy các tác vụ đang chờ ban đầu
    for t_path in tasks_to_run:
        await process_single_task(t_path)

    # 2. Nếu bật chế độ chạy ngầm liên tục 24h cho Colab Pro+
    if RUN_CONTINUOUS_DAEMON:
        print("\n🔄 [COLAB PRO+ 24H DAEMON] Bắt đầu chế độ túc trực quét hàng đợi mỗi 15s...")
        print("💡 Bạn có thể tắt app, đóng tab trình duyệt, Colab Pro+ sẽ chạy ngầm 24h tự động nhận việc!")
        idle_cycles = 0
        while True:
            # Ghi heartbeat để Server Local/Scheduler biết Colab đang trực tuyến
            try:
                hb_data = {
                    "timestamp": time.time(),
                    "time_str": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "gpu": ENCODER,
                    "status": "listening"
                }
                with open(DRIVE_FARM_DIR / "colab_heartbeat.json", "w", encoding="utf-8") as hbf:
                    json.dump(hb_data, hbf)
            except Exception:
                pass

            pending_now = sorted(list(PENDING_DIR.glob("task_*.json")))
            if pending_now:
                idle_cycles = 0
                print(f"\n📥 [Colab Pro+ 24H] Phát hiện {len(pending_now)} tác vụ mới! Bắt đầu dựng...")
                for nt in pending_now:
                    await process_single_task(nt)
            else:
                idle_cycles += 1
                if idle_cycles % 20 == 0:
                    print(f"☕ [Túc trực {time.strftime('%H:%M:%S')}] Hàng đợi trống, GPU sẵn sàng nhận kịch bản mới...")

            await asyncio.sleep(15)

await run_worker_daemon()


In [ ]:
# [BƯỚC 7] TỰ ĐỘNG TẮT GPU & NGẮT KẾT NỐI (TIẾT KIỆM COMPUTE UNITS PRO+ TUYỆT ĐỐI)
# 🛡️ BẢO VỆ TÀI KHOẢN PRO+: Khi render xong (hoặc không có task), Colab sẽ tự động ngắt kết nối runtime ngay lập tức!
# Nhờ vậy bạn KHÔNG BỊ TRỪ COMPUTE UNITS khi treo máy, 1 tháng dùng tẹt ga không bao giờ lo hết quota!

print("=" * 75)
print("🛡️ BẢO VỆ QUOTA: TỰ ĐỘNG GIẢI PHÓNG GPU & NGẮT KẾT NỐI RUNTIME...")
print("✅ Toàn bộ video đã được lưu vào Google Drive an toàn.")
print("🤖 Bot Telegram đã/đang nhận video để gửi về nhóm cho bạn.")
print("⚡ Đang tắt GPU và unassign runtime ngay bây giờ (tiêu tốn 0 Compute Units khi nghỉ)...")
print("=" * 75)

if not globals().get("RUN_CONTINUOUS_DAEMON", False):
    print("=" * 75)
    print("🛡️ BẢO VỆ QUOTA: TỰ ĐỘNG GIẢI PHÓNG GPU & NGẮT KẾT NỐI RUNTIME...")
    print("✅ Toàn bộ video đã được lưu vào Google Drive an toàn.")
    print("🤖 Bot Telegram đã/đang nhận video để gửi về nhóm cho bạn.")
    print("⚡ Đang tắt GPU và unassign runtime ngay bây giờ...")
    print("=" * 75)
    import time
    time.sleep(3)
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print(f"Hoàn thành phiên làm việc! (Lưu ý ngoài Colab: {e})")
else:
    print("✨ [COLAB PRO+ 24H] Đang chạy ngầm túc trực 24h, bảo lưu runtime liên tục không ngắt!")
